## Baking subreddit data cleaning and enriching

This notebook cleans the raw r/baking data, inclduing:
- Dropping unneeded columns
- Converting the post date from UTC (ms since epoch) to MM/DD/YYYY
- Doing a fuzzy match to filter for posts that are similar to GBBO technical bakes
- Calculating and adding a column that denotes the distance from the relevant GBBO episode in which the post was made.

In [1]:
import os
import json
import glob
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, utils
from pathlib import Path
from datetime import datetime, timezone
import re

In [2]:
data_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_data_raw.csv")
if data_csv.is_file():
    baking_reddit_df = pd.read_csv(data_csv)
    print(f"file already exists")
else:
    data=[]
    with open ('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_baking_posts.jsonl', 'r') as file:
        for line in file:
            data.append(json.loads(line))
        baking_reddit_df = pd.read_json("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_baking_posts.jsonl", lines=True)

file already exists


In [3]:
columns_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_raw_data_columns.csv")
if columns_csv.is_file():
    print(f"file already exists")
else:
    columns = baking_reddit_df.columns.to_frame(index=False)
    columns.to_csv('baking_reddit_raw_data_columns.csv', index=False)

file already exists


In [4]:
baking_reddit_df

,Unnamed: 0,created_utc,thumbnail,title,url
0,0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...
1,1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw
2,2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...
3,3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...
4,4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...
...,...,...,...,...,...
528981,528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc
528982,528982,1784676571,default,Biscuits à la Sally,NaN
528983,528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg
528984,528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo


In [5]:
baking_reddit_df_cols_to_keep = [
    'created_utc',
    'thumbnail',
    'title',
    'url',
]

baking_reddit_df_clean = baking_reddit_df[baking_reddit_df_cols_to_keep]
baking_reddit_df_clean.to_csv('baking_reddit_data_complete.csv')
baking_reddit_df_clean

,created_utc,thumbnail,title,url
0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...
1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw
2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...
3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...
4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...
...,...,...,...,...
528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc
528982,1784676571,default,Biscuits à la Sally,NaN
528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg
528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo


In [6]:
baking_reddit_df_clean['date'] = pd.to_datetime(baking_reddit_df_clean['created_utc'], unit='s')
baking_reddit_df_clean['date'] = baking_reddit_df_clean['date'].dt.strftime('%m/%d/%Y')
baking_reddit_df_clean

,created_utc,thumbnail,title,url,date
0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...,08/01/2010
1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw,08/04/2010
2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...,08/04/2010
3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...,08/04/2010
4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...,08/05/2010
...,...,...,...,...,...
528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc,07/21/2026
528982,1784676571,default,Biscuits à la Sally,NaN,07/21/2026
528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg,07/21/2026
528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo,07/21/2026


In [7]:
baking_reddit_post_titles = baking_reddit_df_clean['title']
baking_reddit_post_titles

0         Come to Cupcake Camp OC, tomorrow. Eat tons of...
1         My junior's cheesecake woes... any advice appr...
2         Chocolate Cake with Chocolate Peanut Butter Fr...
3                              First Attempt at White Bread
4               HELP -- Pastry Cream Butter Cream is BROKEN
                                ...                        
528981    Self Developed Recipe! Walnut Shortbread brown...
528982                                  Biscuits à la Sally
528983                                  Biscuits à la Sally
528984    Low k screwed up the lattice , but it still lo...
528985    Please Share Your Best Banana Bread Recipe As ...
Name: title, Length: 528986, dtype: str

In [8]:
technicals_df = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')
technicals_cols = technicals_df['Technical']
technicals_cols

0                       victoria sandwich
1                                  scones
2                                     cob
3                 mini hot lemon soufflés
4                         cornish pasties
                      ...                
129          lemon and thyme drizzle cake
130    orange and ginger treacle puddings
131                      caterpiller cake
132                       tart aux pommes
133                     lardy cake slices
Name: Technical, Length: 134, dtype: str

In [9]:
#iterate through each technical
matched_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_posts_matched.csv")

if matched_csv.is_file():
    baking_reddit_df = pd.read_csv(matched_csv)
    print(f"file already exists")

else:

    choices = technicals_cols
    titles = baking_reddit_df_clean['title'].tolist()

    score_matrix = process.cdist(titles, choices, scorer=fuzz.ratio)

    best_scores = score_matrix.max(axis=1)
    best_idx = score_matrix.argmax(axis=1)

    baking_reddit_df_clean['bakeoff_score'] = best_scores
    baking_reddit_df_clean['bakeoff_match'] = [choices[i] for i in best_idx]
    baking_reddit_df_clean['bakeoff'] = best_scores > 70

    baking_reddit_df_clean = baking_reddit_df_clean[baking_reddit_df_clean['bakeoff']][['date','title', 'bakeoff_match', 'bakeoff_score']]
    baking_reddit_df_clean.to_csv('baking_reddit_posts_matched.csv')

In [10]:
#add air dates
technical_bakes = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')

baking_reddit_df_clean_dated = pd.merge(baking_reddit_df_clean, technical_bakes, left_on='bakeoff_match', right_on='Technical', how='left')
baking_reddit_df_clean_dated


,date,title,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric
0,12/06/2010,Red Velvet Cupcakes,red velvet cake,76.470589,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0
1,03/16/2011,Need help with a pineapple upside down cake,miniature pineapple upside-down cakes,71.604935,9/22/2020,11,1,cake,battenberg cake,120.0,miniature pineapple upside-down cakes,90.0,celebrity hero cake bust,240.0,2.0
2,04/12/2011,Red velvet cake?,red velvet cake,90.322578,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0
3,07/20/2011,I made English muffins!!,english muffins,80.000000,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN
4,08/07/2011,white chocolate raspberry cupcakes,white chocolate and blackberry cheesecakes,75.324677,10/18/2023,14,4,chocolate,chocolate torte,NaN,white chocolate and blackberry cheesecakes,NaN,edible chocolate box cake,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3639,07/17/2026,Tiramisu Cake Roll,tiramisu cake,70.967743,8/27/2014,5,4,dessert,8 self-saucing puddings,120.0,tiramisu cake,150.0,baked alaska,270.0,NaN
3640,07/18/2026,Cherry Cheesecake,cherry cake,71.428574,8/6/2014,5,1,cake,swiss roll,150.0,cherry cake,120.0,36 classic miniature british cakes,210.0,NaN
3641,07/19/2026,First focaccia,focaccia,72.727272,8/30/2011,2,3,bread,free form loaf,195.0,focaccia,210.0,12 sweet and 12 savoury rolls,300.0,NaN
3642,07/19/2026,Chocolate ale cake,chocolate teacakes,75.675674,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0


In [11]:
baking_reddit_df_clean_dated.drop(columns=['bakeoff_match'])
baking_reddit_df_clean_dated['reddit_post_title'] = baking_reddit_df_clean_dated['title']
baking_reddit_df_clean_dated['reddit_post_date'] = baking_reddit_df_clean_dated['date']
baking_reddit_df_clean_dated

,date,title,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,12/06/2010,Red Velvet Cupcakes,red velvet cake,76.470589,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red Velvet Cupcakes,12/06/2010
1,03/16/2011,Need help with a pineapple upside down cake,miniature pineapple upside-down cakes,71.604935,9/22/2020,11,1,cake,battenberg cake,120.0,miniature pineapple upside-down cakes,90.0,celebrity hero cake bust,240.0,2.0,Need help with a pineapple upside down cake,03/16/2011
2,04/12/2011,Red velvet cake?,red velvet cake,90.322578,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red velvet cake?,04/12/2011
3,07/20/2011,I made English muffins!!,english muffins,80.000000,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,I made English muffins!!,07/20/2011
4,08/07/2011,white chocolate raspberry cupcakes,white chocolate and blackberry cheesecakes,75.324677,10/18/2023,14,4,chocolate,chocolate torte,NaN,white chocolate and blackberry cheesecakes,NaN,edible chocolate box cake,NaN,2.0,white chocolate raspberry cupcakes,08/07/2011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3639,07/17/2026,Tiramisu Cake Roll,tiramisu cake,70.967743,8/27/2014,5,4,dessert,8 self-saucing puddings,120.0,tiramisu cake,150.0,baked alaska,270.0,NaN,Tiramisu Cake Roll,07/17/2026
3640,07/18/2026,Cherry Cheesecake,cherry cake,71.428574,8/6/2014,5,1,cake,swiss roll,150.0,cherry cake,120.0,36 classic miniature british cakes,210.0,NaN,Cherry Cheesecake,07/18/2026
3641,07/19/2026,First focaccia,focaccia,72.727272,8/30/2011,2,3,bread,free form loaf,195.0,focaccia,210.0,12 sweet and 12 savoury rolls,300.0,NaN,First focaccia,07/19/2026
3642,07/19/2026,Chocolate ale cake,chocolate teacakes,75.675674,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate ale cake,07/19/2026


In [12]:
GBBO_reddit_posts_merged = baking_reddit_df_clean_dated.drop(columns=['title', 'date'])
GBBO_reddit_posts_merged

,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,red velvet cake,76.470589,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red Velvet Cupcakes,12/06/2010
1,miniature pineapple upside-down cakes,71.604935,9/22/2020,11,1,cake,battenberg cake,120.0,miniature pineapple upside-down cakes,90.0,celebrity hero cake bust,240.0,2.0,Need help with a pineapple upside down cake,03/16/2011
2,red velvet cake,90.322578,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red velvet cake?,04/12/2011
3,english muffins,80.000000,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,I made English muffins!!,07/20/2011
4,white chocolate and blackberry cheesecakes,75.324677,10/18/2023,14,4,chocolate,chocolate torte,NaN,white chocolate and blackberry cheesecakes,NaN,edible chocolate box cake,NaN,2.0,white chocolate raspberry cupcakes,08/07/2011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3639,tiramisu cake,70.967743,8/27/2014,5,4,dessert,8 self-saucing puddings,120.0,tiramisu cake,150.0,baked alaska,270.0,NaN,Tiramisu Cake Roll,07/17/2026
3640,cherry cake,71.428574,8/6/2014,5,1,cake,swiss roll,150.0,cherry cake,120.0,36 classic miniature british cakes,210.0,NaN,Cherry Cheesecake,07/18/2026
3641,focaccia,72.727272,8/30/2011,2,3,bread,free form loaf,195.0,focaccia,210.0,12 sweet and 12 savoury rolls,300.0,NaN,First focaccia,07/19/2026
3642,chocolate teacakes,75.675674,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate ale cake,07/19/2026


In [13]:
#calculating days since technical airdate in which reddit post was made

GBBO_reddit_posts_merged['Airdate'] = pd.to_datetime(GBBO_reddit_posts_merged['Airdate'])
GBBO_reddit_posts_merged['reddit_post_date'] = pd.to_datetime(GBBO_reddit_posts_merged['reddit_post_date'])

# GBBO_reddit_posts_merged['days_since_air'] = []

posts = GBBO_reddit_posts_merged['reddit_post_title']

for post in posts:
    GBBO_reddit_posts_merged['days_since_air'] = GBBO_reddit_posts_merged['Airdate'] - GBBO_reddit_posts_merged['reddit_post_date']

GBBO_reddit_posts_merged.to_csv('GBBO_reddit_posts_merged.csv', index=False)